## ERFD Fake News Detection - Quick Inference Demo
This notebook demonstrates how to automatically download the pre-trained ERFD model and evaluate it on the test set without any manual setup.

In [ ]:
# 1. Clone repo and install dependencies
!git clone https://github.com/HMXHY/ERFD.git
%cd ERFD
!pip install -r requirements.txt gdown
import sys
sys.path.append('.')

In [ ]:
# 2. Download Data and Checkpoints from Google Drive
!gdown --id 1GF3yC8hKWIDwrxJvZYgTHkzsv8BBhgOH
!unzip -o ERFD_demo_files.zip
print("Data and checkpoints successfully loaded!")

In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore")
from src.ERFD import Classifier, create_eval_loader, test_ouput
from utils.load_graphdata import load_origindata_test
from result_output.log_result import output_metrics_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Mock Arguments for Jupyter Environment
class Args:
    dataset_name = 'politifact'
    hidden_dim = 768
    freq_dim = 4
    attn_heads = 1
    dropout_num = 0.4
    max_len = 512
    batch_size = 4
args = Args()

# 4. Load Test Data
print("Loading Test Data...")
test_input_ids, test_masks, test_label = load_origindata_test(args.dataset_name)
test_loader = create_eval_loader(test_input_ids['O'], test_masks['O'], test_label, args.max_len, args.batch_size)

# 5. Load Pre-trained Model
print("Loading Pre-trained Model Parameters...")
model = Classifier(args.hidden_dim, args.freq_dim, args.attn_heads, args.dropout_num).to(device)
model.load_state_dict(torch.load('checkpoints/ERFD/politifact_iter0.m', map_location=device))
model.eval()

# 6. Run Evaluation
print("Running Inference on Test Set...")
y_test, y_pred = test_ouput(test_loader, model)
output_metrics_metrics(y_test, y_pred, 'Origin Test Set')